In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

side_effects = [] 

class State(TypedDict):
    refund: float
    outcome: str
    
def review(state: State) -> dict:
    side_effects.append("review node started")
    answer = interrupt({"question": f"Approve refund of ${state['refund']}? (yes/no)"})
    return {"outcome": "approved" if answer.lower() == "yes" else "rejected"}

g = StateGraph(State)
g.add_node("review", review)
g.add_edge(START, "review")
g.add_edge("review", END)

graph = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable":{"thread_id": "t1"}}

In [2]:
first = graph.invoke({"refund": 100.0, "outcome": ""}, cfg)
print("1) first call returned :", first)
print("2) paused? next node   :", graph.get_state(cfg).next)

1) first call returned : {'refund': 100.0, 'outcome': '', '__interrupt__': [Interrupt(value={'question': 'Approve refund of $100.0? (yes/no)'}, id='b26dbae1be280932cf1fbb8f7693f56a', response_schema=None)]}
2) paused? next node   : ('review',)


In [3]:
final = graph.invoke(Command(resume="yes"), cfg)
print("3) after resume        :", final)
print("4) side effects log    :", side_effects)

3) after resume        : {'refund': 100.0, 'outcome': 'approved'}
4) side effects log    : ['review node started', 'review node started']


Fixed version when the things before the resume are not executed again:

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

effects = []

class State(TypedDict):
    out: str
    
def review(state: State) -> dict:
    answer = interrupt("approve?")     # nothing above this line has side effects
    effects.append("email sent")       # runs once, after the human has answered
    return {"out": answer}

g = StateGraph(State)
g.add_node("review", review)
g.add_edge(START, "review"); g.add_edge("review", END)
graph = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "x"}}

graph.invoke({"out": ""}, cfg)
graph.invoke(Command(resume="yes"), cfg)
print("side effects:", effects)

side effects: ['email sent']


with sqllite to persist the off and on

In [6]:
import sqlite3, sys
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    refund: float
    outcome: str

def review(state):
    answer = interrupt({"question": f"Approve refund of ${state['refund']}?"})
    return {"outcome": "approved" if answer == "yes" else "rejected"}

def build(saver):
    g = StateGraph(State)
    g.add_node("review", review)
    g.add_edge(START, "review"); g.add_edge("review", END)
    return g.compile(checkpointer=saver)


In [7]:
mode = "start"
conn = sqlite3.connect("runs.db", check_same_thread=False)
graph = build(SqliteSaver(conn))
cfg = {"configurable": {"thread_id": "ticket-42"}}
if mode == "start":
    out = graph.invoke({"refund": 250.0}, cfg)
    print("PROCESS 1: paused, next =", graph.get_state(cfg).next)
else:
    print("PROCESS 2: state found on disk, next =", graph.get_state(cfg).next)
    print("PROCESS 2: result =", graph.invoke(Command(resume="yes"), cfg))

PROCESS 1: paused, next = ('review',)


In [8]:
mode = "resume"
conn = sqlite3.connect("runs.db", check_same_thread=False)
graph = build(SqliteSaver(conn))
cfg = {"configurable": {"thread_id": "ticket-42"}}
if mode == "start":
    out = graph.invoke({"refund": 250.0}, cfg)
    print("PROCESS 1: paused, next =", graph.get_state(cfg).next)
else:
    print("PROCESS 2: state found on disk, next =", graph.get_state(cfg).next)
    print("PROCESS 2: result =", graph.invoke(Command(resume="yes"), cfg))

PROCESS 2: state found on disk, next = ('review',)
PROCESS 2: result = {'refund': 250.0, 'outcome': 'approved'}
